# Cálculo dos custos de conexão dos latifúndios do Brasil

## Imports

In [1]:
import os
import pandas as pd
import geopandas as gpd
import gcsfs
from dotenv import find_dotenv, load_dotenv


## Paths

In [2]:
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

project_root = os.path.dirname(dotenv_path)

KEY_PATH_RELATIVE = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if KEY_PATH_RELATIVE:
    KEY_PATH_ABSOLUTE = os.path.join(project_root, KEY_PATH_RELATIVE)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH_ABSOLUTE
    print(f"Credenciais carregadas com sucesso de: {KEY_PATH_ABSOLUTE}")
else:
    print(
        "Aviso: A variável GOOGLE_APPLICATION_CREDENTIALS não foi encontrada no arquivo .env"
    )
    print("O acesso ao GCS pode falhar.")

BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
DIR_RAW_GRID = "../data/raw/electrical_grid/"
CANDIDATOS_GCS_PATH = f"gs://{BUCKET_NAME}/processed/candidatos_solares_brasil_limpo.gpkg"
TEMP_FILE = "temp_candidatos_brasil.gpkg"

Credenciais carregadas com sucesso de: g:\BackupC\Faculdade\UFRJ\TCC\analise-geografica-energia-solar-brasil\.secrets/tcc-usinas-solares-brasil-29b2750a20f8.json


## Carregamento de Dados

In [ ]:
fs = gcsfs.GCSFileSystem()
fs.get(CANDIDATOS_GCS_PATH, TEMP_FILE)
candidatos_gdf = gpd.read_file(TEMP_FILE, engine="fiona")
os.remove(TEMP_FILE)
print(f"Candidatos carregados: {len(candidatos_gdf):,} propriedades.")

subestacoes_gdf = gpd.read_file(os.path.join(DIR_RAW_GRID, "Subestações_-_Base_Existente.shp"))
linhas_gdf = gpd.read_file(os.path.join(DIR_RAW_GRID, "Linhas_de_Transmissão_-_Base_Existente.shp"))
print("Dados carregados com sucesso!")

-> Candidatos carregados: 419,133 propriedades.
Dados carregados com sucesso!


## Processamento dos dados

### Remoção de linhas de baixa tensão

In [4]:
linhas_gdf["Tensao_num"] = pd.to_numeric(linhas_gdf["Tensao"], errors="coerce")
linhas_alta_tensao = linhas_gdf[linhas_gdf["Tensao"] >= 230].copy()

### Reprojeção dos dados para cálculo de distâncias

In [5]:
CRS_NACIONAL_METRICO = "EPSG:5880"
candidatos_proj = candidatos_gdf.to_crs(CRS_NACIONAL_METRICO)
subestacoes_proj = subestacoes_gdf.to_crs(CRS_NACIONAL_METRICO)
linhas_proj = linhas_alta_tensao.to_crs(CRS_NACIONAL_METRICO)

## Cálculos

### Distância da rede elétrica

In [7]:
# Juntando geometrias de subestações (Pontos) e linhas (Linhas) num só GeoDataFrame
infra_geom = pd.concat([
    subestacoes_proj[['geometry']], 
    linhas_proj[['geometry']]
], ignore_index=True)

infra_gdf = gpd.GeoDataFrame(infra_geom, geometry='geometry', crs=CRS_NACIONAL_METRICO)

In [8]:
# Extraindo centróides dos latifúndios para cálculo de distâncias
candidatos_centroides = candidatos_proj.copy()
candidatos_centroides['geometry'] = candidatos_centroides.geometry.centroid

In [9]:
# O sjoin_nearest encontra a geometria da infraestrutura mais próxima e retorna a distância
candidatos_com_distancia = gpd.sjoin_nearest(
    candidatos_centroides, 
    infra_gdf, 
    how='left', 
    distance_col='distancia_metros'
)

In [10]:
# Remoção de possíveis duplicatas (duas linhas com esma distância do terreno)
candidatos_com_distancia = candidatos_com_distancia[~candidatos_com_distancia.index.duplicated(keep='first')]

### Custo de conexão

In [11]:
# Conversão de metros pra km
candidatos_com_distancia['distancia_km'] = candidatos_com_distancia['distancia_metros'] / 1000

In [21]:
# PREMISSA DE CUSTO: R$ 1.500.000 por Quilômetro de Linha de Transmissão construída (Valor estimado base EPE)
CUSTO_POR_KM = 1500000 
candidatos_com_distancia['custo_conexao_rs'] = candidatos_com_distancia['distancia_km'] * CUSTO_POR_KM

## Geração do input do solver (matriz)

In [ ]:
# Preparando DataFrame final para solver de otimização
df_solver = pd.DataFrame(candidatos_com_distancia.drop(columns='geometry'))

colunas_solver = [
    'id_imovel', 'nome_imovel', 'uf_municip',
    'area_util_ha', 'potencial_mw', 
    'distancia_km', 'custo_conexao_rs'
]

colunas_presentes = [col for col in colunas_solver if col in df_solver.columns]
df_solver_final = df_solver[colunas_presentes].copy()
df_solver_final = df_solver_final.sort_values(by='custo_conexao_rs').reset_index(drop=True)

In [22]:
df_solver_final.head(5)

,id_imovel,nome_imovel,area_util_ha,potencial_mw,distancia_km,custo_conexao_rs
0,9080701007146,FAZENDA MIRANTE - Fazenda Mirante ? Matrícula ...,155.725294,43.257026,0.000037,55.518734
1,9500769614694,FAZENDA DO MILAGRE - Parte 2,187.416871,52.060242,0.000054,81.333365
2,6200762799007,Fazenda Santa Barbara ? Gleba A2 - Fazenda San...,342.405903,95.112751,0.000092,137.654546
3,9011641816846,LOTE Nº 10 - Parte 1,358.740786,99.650218,0.000126,189.240047
4,2290750025930,FAZENDA MATARY - Faz. Matary,176.237030,48.954731,0.000143,214.839764


In [ ]:
# Salvando resultado final em CSV e enviando para GCS
TEMP_CSV = "temp_solver_input_mg.csv"
df_solver_final.to_csv(TEMP_CSV, index=False, sep=';', decimal=',')

SOLVER_OUTPUT_PATH = f"gs://{BUCKET_NAME}/solver_inputs/candidatos_solver_brasil.csv"
fs.put(TEMP_CSV, SOLVER_OUTPUT_PATH)
os.remove(TEMP_CSV)

print(f"Arquivo salvo com sucesso na nuvem: {SOLVER_OUTPUT_PATH}")

-> Arquivo salvo com sucesso na nuvem: gs://tcc-usinas-solares-brasil-dados/solver_inputs/candidatos_solver_brasil.csv
